# GasRisk AI CSV inventory check

This notebook checks the CSV files currently stored under `Data/` without modifying them. It uses only the Python standard library so it can run before project dependencies are installed.

Checks:
- file list and row count
- column names and row-width consistency
- blank values by column
- date/year ranges
- replacement character (`U+FFFD`) counts that indicate text loss

In [12]:
from pathlib import Path
import csv

project_root = Path.cwd()
if not (project_root / "Data").exists():
    project_root = project_root.parent

data_dir = project_root / "Data" / "Raw"
csv_files = sorted(data_dir.rglob("*.csv"))
print(f"project_root: {project_root}")
print(f"csv files: {len(csv_files)}")

project_root: /Users/namduhus/workplace/02_GasRisk_AI
csv files: 9


In [13]:
def read_csv(path):
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            text = raw.decode(encoding)
            rows = list(csv.reader(text.splitlines()))
            return text, rows, encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", raw, 0, 1, "unsupported CSV encoding")


for path in csv_files:
    text, rows, encoding = read_csv(path)
    relative_path = path.relative_to(project_root)
    print(f"{relative_path} | encoding={encoding} | rows={max(len(rows) - 1, 0):,}")

Data/Raw/01_한국가스안전공사_가스사고 현황(월별 원인별)_20251201.csv | encoding=cp949 | rows=158
Data/Raw/02_한국가스안전공사_ 가스사고 신고 접수 현황_20260323.csv | encoding=cp949 | rows=16,778
Data/Raw/03_한국가스안전공사_국내 가스시설 현황_20250919.csv | encoding=cp949 | rows=17
Data/Raw/04_한국가스안전공사_수소충전소 현황_20260131.csv | encoding=cp949 | rows=273
Data/Raw/04_한국가스안전공사_전국 LPG 충전소 현황_20251127.csv | encoding=cp949 | rows=1,977
Data/Raw/04_한국가스안전공사_전국 도시가스충전소 현황_20260205.csv | encoding=cp949 | rows=191
Data/Raw/05_한국가스안전공사_도시가스특정사용시설 통계_20250915.csv | encoding=cp949 | rows=29
Data/Raw/09_한국가스공사_판매현황_20241231.csv | encoding=cp949 | rows=25
Data/Raw/10_한국가스공사_도시가스 국가수요실적 및 비중_20201231.csv | encoding=cp949 | rows=34


In [14]:
def inspect_csv(path):
    text, rows, encoding = read_csv(path)
    header = rows[0] if rows else []
    data_rows = rows[1:]
    widths = sorted({len(row) for row in rows})
    blanks = {
        column: sum(index >= len(row) or not row[index].strip() for row in data_rows)
        for index, column in enumerate(header)
    }
    ranges = {}
    for index, column in enumerate(header):
        column_lower = column.lower()
        if "date" in column_lower or column_lower in {"year", "month"}:
            values = [row[index].strip() for row in data_rows if index < len(row) and row[index].strip()]
            if values:
                ranges[column] = (min(values), max(values))
    return {
        "path": path.relative_to(project_root),
        "rows": len(data_rows),
        "encoding": encoding,
        "header": header,
        "widths": widths,
        "blanks": blanks,
        "ranges": ranges,
        "replacement_chars": text.count("\ufffd"),
    }


summaries = [inspect_csv(path) for path in csv_files]
for summary in summaries:
    print(f"\n[{summary['path']}]")
    print(f"encoding: {summary['encoding']}")
    print(f"rows: {summary['rows']:,}")
    print(f"columns ({len(summary['header'])}): {summary['header']}")
    print(f"row widths: {summary['widths']}")
    print(f"blank values: {summary['blanks']}")
    print(f"date/year ranges: {summary['ranges']}")
    print(f"replacement characters (U+FFFD): {summary['replacement_chars']:,}")


[Data/Raw/01_한국가스안전공사_가스사고 현황(월별 원인별)_20251201.csv]
encoding: cp949
rows: 158
columns (10): ['사고연도', '사고월', '가스구분', '공급자취급부주의', '기타(1-3급)', '사용자취급부주의', '시설미비', '제품노후(고장)', '타공사', '교통사고']
row widths: [10]
blank values: {'사고연도': 0, '사고월': 0, '가스구분': 0, '공급자취급부주의': 0, '기타(1-3급)': 0, '사용자취급부주의': 0, '시설미비': 0, '제품노후(고장)': 0, '타공사': 0, '교통사고': 0}
date/year ranges: {}
replacement characters (U+FFFD): 0

[Data/Raw/02_한국가스안전공사_ 가스사고 신고 접수 현황_20260323.csv]
encoding: cp949
rows: 16,778
columns (10): ['연번', '사고번호', '접수지사', '접수일자', '접수요일', '처리지사', '사고일자', '신고자 소속', '주소', '사고가스']
row widths: [10]
blank values: {'연번': 0, '사고번호': 0, '접수지사': 0, '접수일자': 0, '접수요일': 0, '처리지사': 0, '사고일자': 0, '신고자 소속': 0, '주소': 4, '사고가스': 10}
date/year ranges: {}
replacement characters (U+FFFD): 0

[Data/Raw/03_한국가스안전공사_국내 가스시설 현황_20250919.csv]
encoding: cp949
rows: 17
columns (25): ['시도지역', '고압가스특정제조', '고압가스일반제조', '고압가스냉동제조', '고압가스충전', '고압가

In [15]:
print("Files that still contain replacement characters (U+FFFD):")
damaged = [summary for summary in summaries if summary["replacement_chars"]]
for summary in damaged:
    print(f"- {summary['path']}: {summary['replacement_chars']:,}")

print(f"\nresult: {len(damaged)} of {len(summaries)} CSV files contain U+FFFD")

Files that still contain replacement characters (U+FFFD):

result: 0 of 9 CSV files contain U+FFFD


## Interpretation

- A `U+FFFD` count greater than zero means text was replaced before or during a previous conversion. Renaming columns does not recover those values.
- `row widths` should contain one number. Multiple widths indicate malformed CSV rows.
- Weather data can be added later and will appear automatically when stored as CSV below `Data/`.